# C02 — CNN Architectures in PyTorch: From Baseline to ResNet

> **Audience**: PhD students · **Framework**: PyTorch · **Dataset**: FashionMNIST

This notebook builds two architectures from scratch:
1. A baseline CNN to establish the full PyTorch training pipeline
2. ResNet-style skip connections — the single most important architectural
   innovation in deep learning, solving the vanishing-gradient problem

**Why both?**
Understanding the baseline failure mode (vanishing gradients in plain deep networks)
is essential to appreciating *why* skip connections work — not just *that* they work.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

# 1) Data Pipeline

**Dataset**: FashionMNIST — 70 000 greyscale 28×28 images across 10 clothing categories.
Chosen because it is harder than MNIST while still small enough to train quickly on CPU.

**Data augmentation design decisions**
- `RandomHorizontalFlip`: Clothing is symmetric left–right; this doubles effective data.
- `RandomCrop(28, padding=4)`: Shifts the garment slightly; improves position invariance.
- `Normalize(mean, std)`: Centres each pixel to zero mean and unit variance per channel,
  which keeps weight gradients well-scaled during early training.

**Important**: augmentation is applied only to the training set.
The validation set uses only `ToTensor` + `Normalize` to get an unbiased estimate.

In [ ]:
CLASSES = [
    "T-shirt", "Trouser", "Pullover", "Dress", "Coat",
    "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot",
]

# Pixel statistics pre-computed on FashionMNIST training set
MEAN, STD = (0.2860,), (0.3530,)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(28, padding=4),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_set = torchvision.datasets.FashionMNIST(
    root="./data", train=True,  download=True, transform=train_transform
)
val_set = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=val_transform
)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train samples : {len(train_set)}")
print(f"Val   samples : {len(val_set)}")
print(f"Batches/epoch : {len(train_loader)}")

# Visualise a batch
images, labels = next(iter(train_loader))
# (batch_num=128, n_channels=1, h=28, w=28)
grid = torchvision.utils.make_grid(images[:16], nrow=8, normalize=True)
plt.figure(figsize=(12, 3))
plt.imshow(grid.permute(1, 2, 0), cmap="gray")
plt.title("Sample training batch (augmented)")
plt.axis("off")
plt.show()
print([CLASSES[l] for l in labels[:8].tolist()])

# 2) Baseline CNN

**Architecture**: three convolutional blocks followed by a fully-connected classifier.

```
Input (1×28×28)
  ↓ Conv(1→32, 3×3) + BN + ReLU + MaxPool(2)   → 32×14×14
  ↓ Conv(32→64, 3×3) + BN + ReLU + MaxPool(2)  → 64×7×7
  ↓ Conv(64→128, 3×3) + BN + ReLU + MaxPool(2) → 128×3×3
  ↓ Flatten → Linear(1152→256) + ReLU + Dropout(0.5)
  ↓ Linear(256→10)
```

**Design decisions**
- **BatchNorm before ReLU**: normalises layer inputs, stabilises training, allows
  higher learning rates. Note: BN behaves differently in `model.train()` vs `model.eval()`.
- **Dropout(0.5) in the classifier**: regularises the dense layer where overfitting
  concentrates; not applied to conv layers (spatial structure reduces their risk).
- **Bias=True (default)**: BN has its own learnable shift (β), so `bias=False` is
  common in conv layers before BN. Here we keep the default for clarity.

In [ ]:
class BaselineCNN(nn.Module):
    """
    Three-block convolutional network with BatchNorm and Dropout.

    Serves as a clean template for the PyTorch module pattern:
    define layers in __init__, compose them in forward.
    """

    def __init__(self, n_classes: int = 10) -> None:
        super().__init__()

        # Block 1: extract low-level edges and textures
        # (batch_num, 1, 28, 28) → (batch_num, 32, 14, 14)
        self.block1 = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # Block 2: build mid-level part combinations
        # (batch_num, 32, 14, 14) → (batch_num, 64, 7, 7)
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # Block 3: capture high-level semantic patterns
        # (batch_num, 64, 7, 7) → (batch_num, 128, 3, 3)
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # Classifier: flatten then two FC layers with dropout
        # (batch_num, 128*3*3) → (batch_num, n_classes)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, 1, 28, 28) → (batch_num, 128, 3, 3)
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        # (batch_num, 128*3*3) → (batch_num, n_classes)
        return self.classifier(x)


# ── Verify shapes with a dry run ───────────────────────────────────────────────
model_test = BaselineCNN()
x_test = torch.zeros(4, 1, 28, 28)

with torch.no_grad():
    out_test = model_test(x_test)

print(f"Input  : {x_test.shape}")   # (4, 1, 28, 28)
print(f"Output : {out_test.shape}") # (4, 10)

total_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}")

# 3) Training Utilities

**Why separate functions?**
Isolating `train_one_epoch` and `evaluate` into pure functions (model + data → metrics)
makes them reusable, testable, and composable — a pattern essential for research code
where you need to swap models, schedulers, and losses independently.

**Critical PyTorch behaviours**
- `model.train()` enables Dropout and uses the *batch statistics* in BatchNorm.
- `model.eval()` disables Dropout and uses the *running statistics* in BatchNorm.
  Forgetting to call `model.eval()` during validation is a common bug that inflates
  validation accuracy (the model sees the batch mean/std of the val batch, not
  the population mean/std learned from training).
- `@torch.no_grad()` prevents building a computation graph during evaluation,
  saving memory and compute.

In [ ]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    optimizer: optim.Optimizer,
    device: torch.device,
) -> float:
    """
    Runs one full pass over the training data.

    Args:
        model     : Network in train mode after this call
        loader    : Training DataLoader
        criterion : Loss function
        optimizer : Parameter update rule

    Returns:
        mean_loss : Average loss per sample over the epoch
    """
    model.train()  # batch-norm uses batch stats; dropout active
    running_loss = 0.0

    for X, y in loader:
        # (batch_num, 1, 28, 28), (batch_num,)
        X, y = X.to(device), y.to(device)

        # Standard PyTorch training step
        optimizer.zero_grad()

        # (batch_num, 1, 28, 28) → (batch_num, n_classes)
        logits = model(X)

        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()

        # Scale by batch size so the average is over samples, not batches
        running_loss += loss.item() * X.size(0)

    return running_loss / len(loader.dataset)


@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
) -> tuple:
    """
    Evaluates model on a held-out set.

    Returns:
        mean_loss : Average loss per sample
        accuracy  : Fraction of correctly classified samples
    """
    model.eval()  # running-mean/var in BN; dropout disabled
    running_loss, correct = 0.0, 0

    for X, y in loader:
        X, y = X.to(device), y.to(device)

        # (batch_num, n_classes)
        logits = model(X)

        running_loss += criterion(logits, y).item() * X.size(0)
        # argmax over class dimension to get predicted label
        # (batch_num, n_classes) → (batch_num,)
        correct += (logits.argmax(dim=1) == y).sum().item()

    n = len(loader.dataset)
    return running_loss / n, correct / n


def train(
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    n_epochs: int,
    lr: float = 1e-3,
    device: torch.device = DEVICE,
) -> dict:
    """
    Full training loop with cosine LR schedule and history logging.

    Returns:
        history : dict with keys 'train_loss', 'val_loss', 'val_acc'
    """
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    # Cosine schedule: decays LR smoothly from lr to 0 over n_epochs
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    model.to(device)

    for epoch in range(1, n_epochs + 1):
        t_loss              = train_one_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_acc       = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        history["train_loss"].append(t_loss)
        history["val_loss"].append(v_loss)
        history["val_acc"].append(v_acc)

        print(f"Epoch {epoch:02d}/{n_epochs} | "
              f"train_loss={t_loss:.4f} | val_loss={v_loss:.4f} | val_acc={v_acc:.3f}")

    return history


def plot_history(history: dict) -> None:
    """Plots training loss and validation accuracy side by side."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(history["train_loss"], label="train")
    ax1.plot(history["val_loss"],   label="val")
    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()

    ax2.plot(history["val_acc"])
    ax2.set_title("Validation accuracy")
    ax2.set_xlabel("Epoch")
    ax2.set_ylim(0, 1)

    plt.tight_layout()
    plt.show()

# 4) Train Baseline CNN

Training for **3 epochs** as a demonstration.
For publication-quality results, train for 50–100 epochs with a proper schedule.

**Expected behaviour**:
- Loss should decrease monotonically on training data.
- Validation accuracy should reach ~85–88% after full training on FashionMNIST.
- The gap between train_loss and val_loss reveals the degree of overfitting.

In [ ]:
torch.manual_seed(42)
baseline_cnn = BaselineCNN(n_classes=10)
history_baseline = train(baseline_cnn, train_loader, val_loader, n_epochs=3, lr=1e-3)
plot_history(history_baseline)

# 5) The Vanishing Gradient Problem

**Why plain deep networks fail**
In a plain network with L layers, the gradient of the loss with respect to
layer 1 is a product of L Jacobians. If each Jacobian has spectral norm < 1,
the product shrinks exponentially:

$$\|\nabla_{\theta_1} L\| \approx \prod_{l=2}^{L} \|J_l\| \cdot \|\nabla_{\theta_L} L\|$$

The result: early layers receive near-zero gradients and stop learning,
even when more capacity would help.

**Demonstration below**
We build a pathologically deep plain network (20 conv layers, no skip connections)
and measure the gradient norm at each layer after one backward pass.
You will see an exponential decrease as we go from the output toward the input.

In [ ]:
class PlainDeepNet(nn.Module):
    """
    20-layer plain convolutional network with NO skip connections.
    Built to demonstrate the vanishing gradient problem.
    """

    def __init__(self, n_classes: int = 10, n_layers: int = 20) -> None:
        super().__init__()

        # Stem: bring 1-channel input to 32 channels
        # (batch_num, 1, 28, 28) → (batch_num, 32, 28, 28)
        layers = [nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
                  nn.BatchNorm2d(32), nn.ReLU(inplace=True)]

        # Stack n_layers plain conv blocks — all same dimension
        # (batch_num, 32, 28, 28) → (batch_num, 32, 28, 28)  (each block)
        for _ in range(n_layers - 2):
            layers += [
                nn.Conv2d(32, 32, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(32),
                nn.ReLU(inplace=True),
            ]

        self.features = nn.Sequential(*layers)

        # Head: global average pool then linear
        # (batch_num, 32, 28, 28) → (batch_num, 32) → (batch_num, n_classes)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  # → (batch_num, 32, 1, 1)
            nn.Flatten(),             # → (batch_num, 32)
            nn.Linear(32, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, 1, 28, 28) → (batch_num, n_classes)
        return self.head(self.features(x))


def measure_gradient_norms(model: nn.Module, x: torch.Tensor, y: torch.Tensor) -> dict:
    """
    Runs one forward-backward pass and records grad norm per named layer.

    Returns:
        norms : {layer_name: gradient_norm}
    """
    model.train()
    criterion = nn.CrossEntropyLoss()

    logits = model(x)
    loss   = criterion(logits, y)
    loss.backward()

    return {
        name: param.grad.norm().item()
        for name, param in model.named_parameters()
        if param.grad is not None and "weight" in name and "bn" not in name
    }


# Synthetic batch — just need shapes, not real data
torch.manual_seed(0)
x_demo = torch.randn(32, 1, 28, 28)
y_demo = torch.randint(0, 10, (32,))

plain_net = PlainDeepNet(n_layers=20)
grad_norms = measure_gradient_norms(plain_net, x_demo, y_demo)

names  = list(grad_norms.keys())
values = list(grad_norms.values())

# Reverse so the plot reads input-layer (left) to output-layer (right)
names_r  = names[::-1]
values_r = values[::-1]

plt.figure(figsize=(12, 4))
plt.bar(range(len(values_r)), values_r, color="steelblue")
plt.xticks(range(len(names_r)), [n.split(".")[0] for n in names_r], rotation=90, fontsize=8)
plt.yscale("log")
plt.xlabel("Layer (input → output)")
plt.ylabel("Gradient norm (log scale)")
plt.title("Vanishing gradients in a 20-layer plain network")
plt.tight_layout()
plt.show()

print(f"Output layer gradient norm : {values_r[-1]:.2e}")
print(f"Input  layer gradient norm : {values_r[0]:.2e}")
print(f"Attenuation ratio          : {values_r[-1]/max(values_r[0], 1e-12):.1f}x")

# 6) ResNet Identity Block

**The key insight of ResNet (He et al., 2016)**

Instead of learning a mapping $H(x)$, a residual block learns the *residual*
$F(x) = H(x) - x$, so that the full mapping becomes:

$$H(x) = F(x) + x$$

**Why this helps gradient flow**

Differentiating through the skip connection gives:

$$\frac{\partial L}{\partial x} = \frac{\partial L}{\partial H} \cdot \left(1 + \frac{\partial F}{\partial x}\right)$$

The `+1` term means the gradient of the loss *always* flows back through
the skip path — even if $F$'s gradient vanishes, the signal is never fully lost.

**Identity block** — used when the input and output dimensions are the same.
The input is added directly to the output with no transformation.

In [ ]:
class IdentityBlock(nn.Module):
    """
    Residual block for when input and output have identical dimensions.

    Architecture (pre-activation style):
        X → Conv(3×3) → BN → ReLU → Conv(3×3) → BN → (+X) → ReLU

    Design decision — BN before the skip-connection addition:
    Adding before the final ReLU allows the residual path to produce
    negative values, which are only clipped after addition. This gives
    more expressive range than placing ReLU before the addition.
    """

    def __init__(self, channels: int) -> None:
        super().__init__()

        # Both conv layers maintain spatial resolution and channel count
        # (batch_num, channels, h, w) → (batch_num, channels, h, w)
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = x  # save input for the skip connection

        # First conv: extract new features
        # (batch_num, channels, h, w) → (batch_num, channels, h, w)
        out = F.relu(self.bn1(self.conv1(x)))

        # Second conv: refine — no ReLU yet so residual can be negative
        # (batch_num, channels, h, w) → (batch_num, channels, h, w)
        out = self.bn2(self.conv2(out))

        # Skip connection: add input to output before activation
        # If this block should learn identity, it just drives F(x) → 0
        # (batch_num, channels, h, w) + (batch_num, channels, h, w)
        out = out + identity

        return F.relu(out)


# ── Shape verification ─────────────────────────────────────────────────────────
ib = IdentityBlock(channels=64)
x_ib = torch.zeros(4, 64, 14, 14)

with torch.no_grad():
    out_ib = ib(x_ib)

print(f"IdentityBlock  in : {x_ib.shape}")   # (4, 64, 14, 14)
print(f"IdentityBlock out : {out_ib.shape}")  # (4, 64, 14, 14)  ← unchanged

# 7) ResNet Convolutional Block

**When dimensions change**

The identity path requires the skip connection tensor to have the same shape
as the residual output. When the number of channels increases (or spatial
dimensions decrease due to stride), the skip connection must be *projected*
using a 1×1 convolution.

This projection is a learned linear transformation — not a fixed operation.
The network learns the best way to map the old feature space into the new one.

In [ ]:
class ConvBlock(nn.Module):
    """
    Residual block that can change both channel count and spatial resolution.

    Used at the start of each ResNet stage to:
    - Double the number of channels (32→64, 64→128, ...)
    - Halve the spatial dimensions via stride=2

    Architecture:
        Main path  : X → Conv(s) → BN → ReLU → Conv(1) → BN
        Short path : X → Conv1×1(s) → BN
        Output     : ReLU(main + short)
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int = 2) -> None:
        super().__init__()

        # Main path — first conv may downsample spatially via stride
        # (batch_num, in_ch, h, w) → (batch_num, out_ch, h//stride, w//stride)
        self.conv1 = nn.Conv2d(in_channels,  out_channels, kernel_size=3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)

        # (batch_num, out_ch, h//stride, w//stride) → (batch_num, out_ch, h//stride, w//stride)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3,
                               stride=1, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)

        # Shortcut projection: 1×1 conv aligns channels and spatial size
        # (batch_num, in_ch, h, w) → (batch_num, out_ch, h//stride, w//stride)
        self.shortcut = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1,
                      stride=stride, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Main path
        # (batch_num, in_ch, h, w) → (batch_num, out_ch, h//stride, w//stride)
        out  = F.relu(self.bn1(self.conv1(x)))
        out  = self.bn2(self.conv2(out))

        # Projected skip — matches out dimensions
        # (batch_num, in_ch, h, w) → (batch_num, out_ch, h//stride, w//stride)
        skip = self.shortcut(x)

        return F.relu(out + skip)


# ── Shape verification ─────────────────────────────────────────────────────────
cb = ConvBlock(in_channels=32, out_channels=64, stride=2)
x_cb = torch.zeros(4, 32, 28, 28)

with torch.no_grad():
    out_cb = cb(x_cb)

print(f"ConvBlock  in : {x_cb.shape}")   # (4, 32, 28, 28)
print(f"ConvBlock out : {out_cb.shape}") # (4, 64, 14, 14)  ← halved spatial, doubled channels

# 8) Mini ResNet for FashionMNIST

**Architecture for 28×28 greyscale input**

```
Input (1×28×28)
  ↓ Stem: Conv(1→32, 3×3) + BN + ReLU              → 32×28×28
  ↓ Stage 1: ConvBlock(32→64, s=2) + IdentityBlock  → 64×14×14
  ↓ Stage 2: ConvBlock(64→128, s=2) + IdentityBlock → 128×7×7
  ↓ Stage 3: ConvBlock(128→256, s=2) + IdentityBlock→ 256×3×3
  ↓ GlobalAvgPool                                    → 256
  ↓ Linear(256→10)
```

**Why global average pooling (GAP) instead of flatten + FC?**
GAP reduces each feature map to a single number (its spatial mean).
This gives:
- **Translation invariance** — response doesn't depend on where in the feature map
  the pattern appears.
- **Drastically fewer parameters** — a 256×7×7 feature map flattened
  would give 12 544 inputs to the FC layer; GAP gives 256.
- **Class activation maps** — GAP enables CAM visualisation (used in c10).

In [ ]:
class MiniResNet(nn.Module):
    """
    Small ResNet adapted for 28×28 single-channel input (FashionMNIST).

    Uses the same structural ideas as ResNet-18 but scaled down so it
    trains quickly on a CPU in a few epochs.
    """

    def __init__(self, n_classes: int = 10) -> None:
        super().__init__()

        # Stem: initial feature extraction without downsampling
        # (batch_num, 1, 28, 28) → (batch_num, 32, 28, 28)
        self.stem = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        # Stage 1: double channels, halve spatial
        # (batch_num, 32, 28, 28) → (batch_num, 64, 14, 14)
        self.stage1 = nn.Sequential(
            ConvBlock(32, 64, stride=2),
            IdentityBlock(64),
        )

        # Stage 2: double channels, halve spatial
        # (batch_num, 64, 14, 14) → (batch_num, 128, 7, 7)
        self.stage2 = nn.Sequential(
            ConvBlock(64, 128, stride=2),
            IdentityBlock(128),
        )

        # Stage 3: double channels, halve spatial
        # (batch_num, 128, 7, 7) → (batch_num, 256, 3, 3)
        self.stage3 = nn.Sequential(
            ConvBlock(128, 256, stride=2),
            IdentityBlock(256),
        )

        # Head: global average pool collapses (H, W) to scalar per channel
        # (batch_num, 256, 3, 3) → (batch_num, 256) → (batch_num, n_classes)
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(256, n_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # (batch_num, 1, 28, 28) → (batch_num, 32, 28, 28)
        x = self.stem(x)
        # (batch_num, 32, 28, 28) → (batch_num, 64, 14, 14)
        x = self.stage1(x)
        # (batch_num, 64, 14, 14) → (batch_num, 128, 7, 7)
        x = self.stage2(x)
        # (batch_num, 128, 7, 7) → (batch_num, 256, 3, 3)
        x = self.stage3(x)
        # (batch_num, 256, 3, 3) → (batch_num, n_classes)
        return self.head(x)


# ── Shape dry-run ──────────────────────────────────────────────────────────────
torch.manual_seed(0)
resnet = MiniResNet()
x_r    = torch.zeros(4, 1, 28, 28)

with torch.no_grad():
    out_r = resnet(x_r)

print(f"Input  : {x_r.shape}")   # (4, 1, 28, 28)
print(f"Output : {out_r.shape}") # (4, 10)
print(f"Params : {sum(p.numel() for p in resnet.parameters() if p.requires_grad):,}")

In [ ]:
torch.manual_seed(42)
resnet_model  = MiniResNet(n_classes=10)
history_resnet = train(resnet_model, train_loader, val_loader, n_epochs=3, lr=1e-3)
plot_history(history_resnet)

## 8.1 Compare Gradient Flow: ResNet vs Plain Network

After training, we compare gradient norms in the ResNet's early layers to those
in the PlainDeepNet from Section 5.
The ResNet's skip connections should show significantly less attenuation.

In [ ]:
# Measure gradients in the trained ResNet
resnet_model.train()
criterion_check = nn.CrossEntropyLoss()
X_chk, y_chk = next(iter(train_loader))
X_chk, y_chk = X_chk.to(DEVICE), y_chk.to(DEVICE)

resnet_model.to(DEVICE)
resnet_model.zero_grad()
loss_chk = criterion_check(resnet_model(X_chk), y_chk)
loss_chk.backward()

resnet_norms = {
    name: param.grad.norm().item()
    for name, param in resnet_model.named_parameters()
    if param.grad is not None and "weight" in name and "bn" not in name
}

names_rn  = list(resnet_norms.keys())[::-1]
values_rn = list(resnet_norms.values())[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=False)

axes[0].bar(range(len(values_r)), values_r, color="coral")
axes[0].set_yscale("log")
axes[0].set_title("Plain 20-layer (no skip)")
axes[0].set_xlabel("Layer index (input→output)")
axes[0].set_ylabel("Gradient norm")

axes[1].bar(range(len(values_rn)), values_rn, color="steelblue")
axes[1].set_yscale("log")
axes[1].set_title("MiniResNet (with skip)")
axes[1].set_xlabel("Layer index (input→output)")

plt.suptitle("Gradient norms: skip connections prevent vanishing")
plt.tight_layout()
plt.show()

# Summary

| Concept | Plain CNN | ResNet |
|---|---|---|
| Gradient flow | Exponential decay | Always ≥ 1 (from skip path) |
| Early-layer learning | Stalls in deep nets | Stable across depth |
| Parameter overhead | None | ~10% extra for projection layers |
| When to use | ≤ 10 layers | Whenever going deeper |

**Key takeaways for research**
- **`model.train()` vs `model.eval()`** must be called correctly; BatchNorm
  is the most common silent correctness bug in research code.
- **Skip connections are universal**: ResNet's insight has been applied to
  transformers (attention residuals), U-Net (encoder–decoder skip), and diffusion
  models. Learning it deeply pays dividends across all areas.
- **Global average pooling** is preferred over flatten + FC for its invariance
  properties and compatibility with class activation map visualisation.